# rayleigh1: Rayleigh-Benard convection, heated from below and cooled from above
pyOM2: for_config/rayleigh1.f90, py_config/rayleigh.py

Non-hydrostatic x-z slice (nh_simple). The heat flux Q enters the lowest and
leaves the uppermost grid cell, buoyancy b = grav*beta_T*(T - 10).

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.nh_simple.model import model

In [ ]:
fac        = 2.0
nx, ny, nz = int(64*fac), 1, int(20*fac)
dx = dz    = 0.5/fac
dt         = 0.25/fac
mix        = 5e-3/fac**2
Q, cp      = 175., 4185.5           # heat flux and heat capacity
grav, beta_T = 9.81, 1.67e-4
nsteps     = 2400                   # 300 s
plot_every = 200

par = dict(nx=nx, ny=ny, nz=nz, dx=dx, dy=dx, dz=dz, dt=dt,
           f0=0., N0=0., Ro=1., dsqr=1., periodic_in_z=False,
           Ah=mix, Av=mix, buoyancy_superbee_advection=True)

In [ ]:
class rayleigh1(model):

    def set_forcing(self, tau):
        h = self.halo
        self.db = OM.add_to_array(self.db, (tau, slice(None), slice(None), h),      grav*beta_T*Q/cp/dz)
        self.db = OM.add_to_array(self.db, (tau, slice(None), slice(None), -h - 1), -grav*beta_T*Q/cp/dz)

In [ ]:
M = rayleigh1(par)
M.setup()
# local coordinates with halo points, arrays are (x,y,z), z = 0 at the surface
h = M.halo
x = (np.arange(M.Nx) - h + M.xs_pe - 0.5)*dx
y = (np.arange(M.Ny) - h + M.ys_pe - 0.5)*dx
z = (np.arange(M.Nz) - h + M.zs_pe - 0.5)*dz - nz*dz
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
M.b = M.apply_bc(OM.np.asarray(grav*beta_T*0.05*np.sin(X/(20*dx)*np.pi)))

In [ ]:
def plot(t):
    clear_output(wait=True)
    T = M.b[h:-h, h, h:-h].T/(grav*beta_T)
    plt.figure(figsize=(9, 3.5))
    plt.pcolormesh(x[h:-h], z[h:-h], T, cmap='RdBu_r')
    plt.colorbar(label='T anomaly [deg C]'); plt.xlabel('x [m]'); plt.ylabel('z [m]')
    plt.title('t = %.0f s' % t); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt)